In [ ]:
!pip install -q chess

In [ ]:
import random
from collections import Counter
import numpy as np
import chess
import matplotlib.pyplot as plt

FIXED_FEN = "8/8/8/4k3/8/8/8/3QK3 w - - 0 1"
N_ACTIONS = 64 * 64

PLANE = {
    (chess.KING, chess.WHITE): 0,
    (chess.QUEEN, chess.WHITE): 1,
    (chess.KING, chess.BLACK): 2
}

In [ ]:
def board_to_tensor(board):
  x = np.zeros((3, 8, 8), dtype=np.float32)
  for square, piece in board.piece_map().items():
    key = (piece.piece_type, piece.color)
    if key not in PLANE:
      raise ValueError(f"Quân không được hỗ trợ trong bài này: {piece}")
    x [PLANE[key], chess.square_rank(square), chess.square_file(square)] = 1.0
  return x


b = chess.Board(FIXED_FEN)
print(b)
x = board_to_tensor(b)
print(x.shape, x.sum(axis=(1,2)))

. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . k . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . Q K . . .
(3, 8, 8) [1. 1. 1.]


In [ ]:
def move_to_index(move):
  return move.from_square * 64 + move.to_square

def index_to_move(index):
  return chess.Move(index // 64, index % 64)

def legal_mask(board):
  mask = np.zeros(N_ACTIONS, dtype=bool)
  for m in board.legal_moves:
    mask[move_to_index(m)] = True
  return mask

m = chess.Move.from_uci("e1e5")
print(move_to_index(m), index_to_move(move_to_index(m)), legal_mask(b).sum())

292 e1e5 21


In [ ]:
def sample_position(rng, level=2, max_tries=10000):
  if level == 0:
    return chess.Board(FIXED_FEN)

  for _ in range(max_tries):
    white_king, white_queen, black_king = rng.sample(range(64),3)
    if level == 1:
      r, f = chess.square_rank(black_king), chess.square_file(black_king)
      if r not in (0, 7) and f not in (0, 7):
        continue
    board = chess.Board(None)
    board.set_piece_at(white_king, chess.Piece(chess.KING, chess.WHITE))
    board.set_piece_at(black_king, chess.Piece(chess.KING, chess.BLACK))
    board.set_piece_at(white_queen, chess.Piece(chess.QUEEN, chess.WHITE))
    board.turn = chess.WHITE

    if not board.is_valid() or board.is_game_over():
      continue
    return board

  raise RuntimeError("Không sinh được vị trí hợp lệ")
rng = random.Random(0)
print(sample_position(rng, level=2))


. . . . . . Q .
. . . k . . . .
. . . . . . . .
. K . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .


In [ ]:
def _center_distance(sq):
    """Khoảng cách Chebyshev từ sq tới tâm bàn cờ (cao = gần biên)."""
    r, f = chess.square_rank(sq), chess.square_file(sq)
    return max(abs(r - 3.5), abs(f - 3.5))

def _chebyshev(sq1, sq2):
    r1, f1 = chess.square_rank(sq1), chess.square_file(sq1)
    r2, f2 = chess.square_rank(sq2), chess.square_file(sq2)
    return max(abs(r1 - r2), abs(f1 - f2))

def _potential(board):
    """
    Hàm potential Φ(s) ∈ [0, 1].
    Giá trị cao = thế trận tốt cho trắng.
    """
    bk = board.king(chess.BLACK)
    wk = board.king(chess.WHITE)
    wq_squares = list(board.pieces(chess.QUEEN, chess.WHITE))
    if not wq_squares:
        return 0.0
    wq = wq_squares[0]

    bk_edge = _center_distance(bk) / 3.5

    king_close = 1.0 - _chebyshev(wk, bk) / 7.0

    queen_king = 1.0 - _chebyshev(wq, bk) / 7.0

    return 0.5 * bk_edge + 0.3 * king_close + 0.2 * queen_king

b_test = chess.Board(FIXED_FEN)
print(f"Potential của vị trí ban đầu: {_potential(b_test):.3f}")

Potential của vị trí ban đầu: 0.286


In [ ]:
class KQKEnv:
  n_actions = N_ACTIONS
  observation_shape = (3,8,8)

  def __init__(self, max_moves=50, win_reward=5.0, step_penalty=-0.01,
               draw_reward=0.0, lose_queen_reward=-0.3, stalemate_reward=-0.5,
               level=0, seed=None, shaping_weight=0.3):
      self.max_moves = max_moves
      self.win_reward = win_reward
      self.step_penalty = step_penalty
      self.draw_reward = draw_reward
      self.lose_queen_reward = lose_queen_reward
      self.stalemate_reward = stalemate_reward
      self.level = level
      self.rng = random.Random(seed)
      self.board = None
      self.moves = 0
      self.shaping_weight = shaping_weight
      self._prev_potential = 0.0

  def _info(self, result):
      return {"legal_mask": legal_mask(self.board), "result": result}

  def _shaped_reward(self, base_reward, terminal):
      """Thêm F = γ·Φ(s') - Φ(s) vào reward (potential-based shaping)."""
      if terminal or self.shaping_weight == 0:
          return base_reward
      phi_new = _potential(self.board)
      F = self.shaping_weight * (phi_new - self._prev_potential)
      self._prev_potential = phi_new
      return base_reward + F

  def _finish(self):
      name = self.board.outcome().termination.name.lower()
      if self.board.is_checkmate() and self.board.turn == chess.BLACK:
            reward = self.win_reward
      elif name == "insufficient_material":
            reward = self.lose_queen_reward
      elif name == "stalemate":
            reward = self.stalemate_reward
      else:
            reward = self.draw_reward
      return board_to_tensor(self.board), reward, True, False, self._info(name)

  def reset(self, seed=None, fen=None, level=None):
      if seed is not None:
        self.rng = random.Random(seed)
      if fen is not None:
        self.board = chess.Board(fen)
      else:
        lvl = self.level if level is None else level
        self.board = sample_position(self.rng, lvl)
      self.moves = 0
      self._prev_potential = _potential(self.board)
      return board_to_tensor(self.board), self._info(None)

  def step(self, action):
      move = index_to_move(int(action))
      if move not in self.board.legal_moves:
        raise ValueError(f"Nước không hợp lệ: {move.uci()}. Hãy che (mask) trước khi chọn action.")

      self.board.push(move)
      self.moves += 1
      if self.board.is_game_over():
        return self._finish()

      reply = self.rng.choice(list(self.board.legal_moves))
      self.board.push(reply)
      if self.board.is_game_over():
        return self._finish()

      base = self.step_penalty
      if self.moves >= self.max_moves:
        return board_to_tensor(self.board), base, False, True, self._info("max_moves")
      shaped = self._shaped_reward(base, False)
      return board_to_tensor(self.board), shaped, False, False, self._info(None)

  def render(self):
      print(self.board)

In [ ]:
def random_policy(env, rng):
  moves = list(env.board.legal_moves)
  return move_to_index(rng.choice(moves))

def greedy_policy(env, rng):
  board = env.board
  moves = list(board.legal_moves)
  for mv in moves:
    board.push(mv)
    mate = board.is_checkmate()
    board.pop()
    if mate:
      return move_to_index(mv)
  return move_to_index(rng.choice(moves))


def play_games(env, policy, n_games = 5000, level = 0, seed = 0):
  rng = random.Random(seed)
  results = Counter()
  lengths, rewards = [], []
  for g in range(n_games):
      env.reset(seed=seed + g, level = level)
      done, n_steps, total = False, 0, 0.0
      while not done:
        action = policy(env, rng)
        _, r, terminated, truncated, info = env.step(action)
        total += r
        n_steps += 1
        done = terminated or truncated
      results[info["result"]] += 1
      lengths.append(n_steps)
      rewards.append(total)
  return {
      "policy": policy.__name__, "level": level, "n_games": n_games,
      "results": dict(results),
      "checkmate_rate": results["checkmate"] / n_games,
      "queen_loss_rate": results["insufficient_material"] / n_games,
      "stalemate_rate": results["stalemate"] / n_games,
      "timeout_rate": results["max_moves"] / n_games,
      "mean_length": float(np.mean(lengths)),
      "mean_reward": float(np.mean(rewards)),
  }

In [ ]:
def run_tests():
    b = chess.Board(FIXED_FEN)
    x = board_to_tensor(b)
    assert x.shape == (3, 8, 8)
    assert x.sum(axis=(1, 2)).tolist() == [1.0, 1.0, 1.0]
    assert x[1, 0, 3] == 1.0
    assert x[0, 0, 4] == 1.0
    assert x[2, 4, 4] == 1.0

    for mv in b.legal_moves:
        assert index_to_move(move_to_index(mv)) == mv
    assert legal_mask(b).sum() == b.legal_moves.count()

    env = KQKEnv(shaping_weight=0.0)
    env.reset(fen="7k/8/5K2/8/8/8/8/6Q1 w - - 0 1")
    _, r, term, trunc, info = env.step(move_to_index(chess.Move.from_uci("g1g7")))
    assert term and not trunc and r == env.win_reward and info["result"] == "checkmate", (r, term, info["result"])

    env.reset(fen="k7/2K5/7Q/8/8/8/8/8 w - - 0 1")
    _, r, term, trunc, info = env.step(move_to_index(chess.Move.from_uci("h6b6")))
    assert term and r == env.stalemate_reward and info["result"] == "stalemate", (r, term, info["result"])

    env.reset()
    try:
        env.step(move_to_index(chess.Move.from_uci("a1a2")))
        raise AssertionError("Lẽ ra phải báo lỗi nước không hợp lệ")
    except ValueError:
        pass

    env2 = KQKEnv(max_moves=1, shaping_weight=0.0)
    env2.reset()
    _, r, term, trunc, info = env2.step(move_to_index(chess.Move.from_uci("e1e2")))
    assert trunc and not term and info["result"] == "max_moves"

    rng = random.Random(123)
    for lvl in (1, 2):
        for _ in range(200):
            bd = sample_position(rng, lvl)
            assert bd.is_valid() and not bd.is_game_over() and bd.turn == chess.WHITE
            if lvl == 1:
                bk = bd.king(chess.BLACK)
                assert chess.square_rank(bk) in (0, 7) or chess.square_file(bk) in (0, 7)

    env_s = KQKEnv(max_moves=50, shaping_weight=0.1)
    env_s.reset(fen=FIXED_FEN)
    p0 = _potential(env_s.board)
    assert 0 <= p0 <= 1, f"Potential ngoài khoảng: {p0}"

    env3 = KQKEnv(max_moves=50, shaping_weight=0.0)
    n_queen_loss = 0
    for g in range(200):
        env3.reset(seed=g, level=0)
        prng = random.Random(g)
        done = False
        while not done:
            _, r, term, trunc, info = env3.step(random_policy(env3, prng))
            done = term or trunc
        if info["result"] == "insufficient_material":
            assert r == env3.lose_queen_reward, r
            n_queen_loss += 1
        elif info["result"] == "checkmate":
            assert r == env3.win_reward, r
    assert n_queen_loss > 0, "Lẽ ra phải có ván mất hậu khi chơi ngẫu nhiên"

    print("Tất cả kiểm tra đều qua.")


if __name__ == "__main__":
    run_tests()

Tất cả kiểm tra đều qua.
